In [1]:
import datasets
from datasets import load_dataset, Dataset
import hashlib
import os
import re
import pandas as pd 
os.chdir("/home/hieunt/VIP")
os.getcwd()

/home/hieunt/miniconda3/envs/verl/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


'/home/hieunt/VIP'

In [2]:
prompt_template = """Solve the following math problem step by step. 
The last line of your response must be exactly of the form:
Answer: X
(where X is the final answer, without quotes or \boxed{}). 
Do not write anything after the answer line.

Below are four examples for format reference.

Example question 1: There are 15 trees in the grove. Grove workers will plant trees in the grove today. After they are done, there will be 21 trees. How many trees did the grove workers plant today?

Response:
There are 15 trees originally. Then there were 21 trees after some more were planted. 
So there must have been 21 - 15 = 6.
Answer: 6

Example question 2: If there are 3 cars in the parking lot and 2 more cars arrive, how many cars are in the parking lot?

Response:
There are originally 3 cars. 2 more cars arrive. 
3 + 2 = 5.
Answer: 5

Example question 3: Leah had 32 chocolates and her sister had 42. If they ate 35, how many pieces do they have left in total?

Response:
Originally, Leah had 32 chocolates. Her sister had 42. 
So in total they had 32 + 42 = 74. 
After eating 35, they had 74 - 35 = 39.
Answer: 39

Example question 4: Jason had 20 lollipops. He gave Denny some lollipops. Now Jason has 12 lollipops. How many lollipops did Jason give to Denny?

Response:
Jason started with 20 lollipops. Then he had 12 after giving some to Denny. 
So he gave Denny 20 - 12 = 8.
Answer: 8

Solve the current question. Remember to put your final answer on its own line after "Answer:".
Current question: {{question}}

Response:
"""

# MATH DAPO & AIME 24

In [35]:
def de_duplicate_dataframe(ds: Dataset) -> Dataset:
    """Deduplicate a HuggingFace Dataset by the first user message content in `prompt`."""
    def _hash_prompt_batch(batch):
        hashes = []
        for p in batch["prompt"]:
            content = p[0]["content"]
            s = "" if content is None else str(content)
            hashes.append(hashlib.sha1(s.encode("utf-8")).hexdigest())
        return {"__prompt_hash__": hashes}

    # 1) Add hash column
    ds_h = ds.map(_hash_prompt_batch, batched=True, batch_size=1000, num_proc=64)

    # 2) First index per hash
    first_index = {}
    for idx, h in enumerate(ds_h["__prompt_hash__"]):
        if h not in first_index:
            first_index[h] = idx

    # 3) Keep first occurrences (preserve order)
    keep_indices = sorted(first_index.values())
    ds_unique = ds_h.select(keep_indices)

    # 4) Drop temp column
    return ds_unique.remove_columns(["__prompt_hash__"])

def switch_prompt(dataframe):
    prefix = "Solve the following math problem step by step. The last line of your response should be of the form Answer: $Answer (without quotes) where $Answer is the answer to the problem.\n\n"
    suffix = '\n\nRemember to put your answer on its own line after "Answer:".'
    pattern = re.compile(re.escape(prefix) + r"(.*?)" + re.escape(suffix), re.DOTALL)

    def switch_prompt_batch(batch):
        prompts_out, questions = [], []
        for p in batch["prompt"]:
            msg0 = dict(p[0])
            content = msg0.get("content", "")
            if not isinstance(content, str):
                content = str(content) if content is not None else ""

            m = pattern.match(content)
            if m:
                q = m.group(1).strip()
                new_content = prompt_template.replace("{{question}}", q)
            else:
                q = content.strip()
                new_content = content

            msg0["content"] = new_content
            new_prompt = list(p)
            new_prompt[0] = msg0

            prompts_out.append(new_prompt)
            questions.append(q)

        return {"prompt": prompts_out, "question": questions}

    return dataframe.map(
        switch_prompt_batch,
        batched=True,
        batch_size=1000,
        num_proc=128,
        desc="Switch prompts"
    )

In [38]:
unprocessed_math_dapo = load_dataset("parquet", data_files="data/dapo-math-17k.parquet")["train"]
ds_unique = de_duplicate_dataframe(unprocessed_math_dapo)
print(len(unprocessed_math_dapo), "->", len(ds_unique))
processed_math_dapo = switch_prompt(ds_unique)
processed_math_dapo[0]["prompt"][0]["content"]

1791700 -> 17398


'Solve the following math problem step by step. \nThe last line of your response must be exactly of the form:\nAnswer: X\n(where X is the final answer, without quotes or \x08oxed{}). \nDo not write anything after the answer line.\n\nBelow are four examples for format reference.\n\nExample question 1: There are 15 trees in the grove. Grove workers will plant trees in the grove today. After they are done, there will be 21 trees. How many trees did the grove workers plant today?\n\nResponse:\nThere are 15 trees originally. Then there were 21 trees after some more were planted. \nSo there must have been 21 - 15 = 6.\nAnswer: 6\n\nExample question 2: If there are 3 cars in the parking lot and 2 more cars arrive, how many cars are in the parking lot?\n\nResponse:\nThere are originally 3 cars. 2 more cars arrive. \n3 + 2 = 5.\nAnswer: 5\n\nExample question 3: Leah had 32 chocolates and her sister had 42. If they ate 35, how many pieces do they have left in total?\n\nResponse:\nOriginally, Lea

In [39]:
processed_math_dapo.push_to_hub("JunHill/Dedup-4shot-DAPO-Math-17k")

Creating parquet from Arrow format: 100%|██████████| 1/1 [00:00<00:00, 10.83ba/s]
Processing Files (1 / 1): 100%|██████████| 8.00MB / 8.00MB, 1.05MB/s  
New Data Upload: 100%|██████████| 8.00MB / 8.00MB, 1.05MB/s  
Uploading the dataset shards: 100%|██████████| 1/1 [00:09<00:00,  9.10s/ shards]


CommitInfo(commit_url='https://huggingface.co/datasets/JunHill/Dedup-4shot-DAPO-Math-17k/commit/4a1201185cabe5ea81b6af85c0405e657a4fa587', commit_message='Upload dataset', commit_description='', oid='4a1201185cabe5ea81b6af85c0405e657a4fa587', pr_url=None, repo_url=RepoUrl('https://huggingface.co/datasets/JunHill/Dedup-4shot-DAPO-Math-17k', endpoint='https://huggingface.co', repo_type='dataset', repo_id='JunHill/Dedup-4shot-DAPO-Math-17k'), pr_revision=None, pr_num=None)

In [ ]:
unprocessed_math_dapo = load_dataset("parquet", data_files="data/dapo-math-17k.parquet")["train"]
ds_unique = de_duplicate_dataframe(unprocessed_math_dapo)
print(len(unprocessed_math_dapo), "->", len(ds_unique))
processed_math_dapo = switch_prompt(ds_unique)
processed_math_dapo[0]["prompt"][0]["content"]

In [42]:
unprocessed_aime_2024 = load_dataset("parquet", data_files="data/aime-2024.parquet")["train"]
ds_unique = de_duplicate_dataframe(unprocessed_aime_2024)
print(len(unprocessed_aime_2024), "->", len(ds_unique))
processed_aime_2024 = switch_prompt(ds_unique)
processed_aime_2024 = processed_aime_2024.map(lambda x: {"data_source": "AIME2024"})
processed_aime_2024[0]


Map (num_proc=64): 100%|██████████| 960/960 [00:02<00:00, 441.64 examples/s]
num_proc must be <= 30. Reducing num_proc to 30 for dataset of size 30.


960 -> 30


Switch prompts (num_proc=30): 100%|██████████| 30/30 [00:01<00:00, 24.18 examples/s]


'Solve the following math problem step by step. \nThe last line of your response must be exactly of the form:\nAnswer: X\n(where X is the final answer, without quotes or \x08oxed{}). \nDo not write anything after the answer line.\n\nBelow are four examples for format reference.\n\nExample question 1: There are 15 trees in the grove. Grove workers will plant trees in the grove today. After they are done, there will be 21 trees. How many trees did the grove workers plant today?\n\nResponse:\nThere are 15 trees originally. Then there were 21 trees after some more were planted. \nSo there must have been 21 - 15 = 6.\nAnswer: 6\n\nExample question 2: If there are 3 cars in the parking lot and 2 more cars arrive, how many cars are in the parking lot?\n\nResponse:\nThere are originally 3 cars. 2 more cars arrive. \n3 + 2 = 5.\nAnswer: 5\n\nExample question 3: Leah had 32 chocolates and her sister had 42. If they ate 35, how many pieces do they have left in total?\n\nResponse:\nOriginally, Lea

In [47]:
processed_aime_2024.push_to_hub("JunHill/4shot-aime24")

Creating parquet from Arrow format: 100%|██████████| 1/1 [00:00<00:00, 322.69ba/s]
Processing Files (1 / 1): 100%|██████████| 37.6kB / 37.6kB, 23.5kB/s  
New Data Upload: 100%|██████████| 37.6kB / 37.6kB, 23.5kB/s  
Uploading the dataset shards: 100%|██████████| 1/1 [00:02<00:00,  2.66s/ shards]


CommitInfo(commit_url='https://huggingface.co/datasets/JunHill/4shot-aime24/commit/54b964624c678b852524a6a4006f9485c9f834f6', commit_message='Upload dataset', commit_description='', oid='54b964624c678b852524a6a4006f9485c9f834f6', pr_url=None, repo_url=RepoUrl('https://huggingface.co/datasets/JunHill/4shot-aime24', endpoint='https://huggingface.co', repo_type='dataset', repo_id='JunHill/4shot-aime24'), pr_revision=None, pr_num=None)

# AIME 25

In [70]:
def convert_to_dapo_format(ds, prompt_template, data_source="AIME2025"):
    def _convert(ex, idx):
        return {
            "data_source": data_source,
            "prompt": [{"content": prompt_template.replace("{{question}}", ex["question"]), "role": "user"}],
            "ability": "AIME",
            "reward_model": {
                "ground_truth": str(ex["answer"]),
                "style": "rule-lighteval/AIME2025"
            },
            "extra_info": {
                "index": idx,
                "raw_problem": ex["question"],
                "split": None
            }
        }

    return ds.map(lambda ex, idx: _convert(ex, idx), with_indices=True)

In [72]:
AIME2025_I = datasets.load_dataset("opencompass/AIME2025", name="AIME2025-I", split="test")
AIME2025_II = datasets.load_dataset("opencompass/AIME2025", name="AIME2025-II", split="test")
aime2025 = datasets.concatenate_datasets([AIME2025_I, AIME2025_II])
aime_processed = convert_to_dapo_format(aime2025, prompt_template=prompt_template, data_source="AIME2025")

Map: 100%|██████████| 30/30 [00:00<00:00, 2654.90 examples/s]


In [73]:
aime_processed.push_to_hub("JunHill/4shot-aime25")

Creating parquet from Arrow format: 100%|██████████| 1/1 [00:00<00:00, 335.71ba/s]
Processing Files (1 / 1): 100%|██████████| 39.4kB / 39.4kB, 19.7kB/s  
New Data Upload: 100%|██████████| 39.4kB / 39.4kB, 19.7kB/s  
Uploading the dataset shards: 100%|██████████| 1/1 [00:02<00:00,  2.99s/ shards]


CommitInfo(commit_url='https://huggingface.co/datasets/JunHill/4shot-aime25/commit/b7e32cd128275095cf358a0792dd04aa466980d6', commit_message='Upload dataset', commit_description='', oid='b7e32cd128275095cf358a0792dd04aa466980d6', pr_url=None, repo_url=RepoUrl('https://huggingface.co/datasets/JunHill/4shot-aime25', endpoint='https://huggingface.co', repo_type='dataset', repo_id='JunHill/4shot-aime25'), pr_revision=None, pr_num=None)

# AMC 

In [27]:
def convert_amc_to_dapo_format(ds, prompt_template, data_source="AMC2025"):
    def _convert(ex, idx):
        return {
            "data_source": data_source,
            "prompt": [{"content": prompt_template.replace("{{question}}", ex["problem"]), "role": "user"}],
            "ability": "AMC",
            "reward_model": {
                "ground_truth": str(ex["answer"]),
                "style": "rule-lighteval/AMC"
            },
            "extra_info": {
                "index": idx,
                "raw_problem": ex["problem"],
                "difficulty": str(ex.get("difficulty", None)),
                "split": None
            }
        }
    dataframe = ds.map(lambda ex, idx: _convert(ex, idx), with_indices=True)
    dataframe = dataframe.remove_columns(["answer", "difficulty"])
    dataframe = dataframe.rename_column("problem", "question")
    return dataframe

In [28]:
from datasets import Dataset

dataset = Dataset.from_file("/home/hieunt/VIP/clone_space/understand-r1-zero/datasets/evaluation_suite/amc/data-00000-of-00001.arrow")

amc = convert_amc_to_dapo_format(dataset, prompt_template, data_source="AMC")


Map: 100%|██████████| 83/83 [00:00<00:00, 3961.39 examples/s]


In [29]:
amc.push_to_hub("JunHill/4shot-amc")

Creating parquet from Arrow format: 100%|██████████| 1/1 [00:00<00:00, 279.62ba/s]
Processing Files (1 / 1): 100%|██████████| 66.7kB / 66.7kB, 33.3kB/s  
New Data Upload: 100%|██████████| 66.7kB / 66.7kB, 33.3kB/s  
Uploading the dataset shards: 100%|██████████| 1/1 [00:03<00:00,  3.16s/ shards]


CommitInfo(commit_url='https://huggingface.co/datasets/JunHill/4shot-amc/commit/766932634d43f81c73f5a389818851b809c13d3f', commit_message='Upload dataset', commit_description='', oid='766932634d43f81c73f5a389818851b809c13d3f', pr_url=None, repo_url=RepoUrl('https://huggingface.co/datasets/JunHill/4shot-amc', endpoint='https://huggingface.co', repo_type='dataset', repo_id='JunHill/4shot-amc'), pr_revision=None, pr_num=None)

In [25]:
amc

Dataset({
    features: ['question', 'data_source', 'prompt', 'ability', 'reward_model', 'extra_info'],
    num_rows: 83
})

In [87]:
aime_2024['prompt']

Column([[{'content': 'Solve the following math problem step by step. \nThe last line of your response must be exactly of the form:\nAnswer: X\n(where X is the final answer, without quotes or \x08oxed{}). \nDo not write anything after the answer line.\n\nBelow are four examples for format reference.\n\nExample question 1: There are 15 trees in the grove. Grove workers will plant trees in the grove today. After they are done, there will be 21 trees. How many trees did the grove workers plant today?\n\nResponse:\nThere are 15 trees originally. Then there were 21 trees after some more were planted. \nSo there must have been 21 - 15 = 6.\nAnswer: 6\n\nExample question 2: If there are 3 cars in the parking lot and 2 more cars arrive, how many cars are in the parking lot?\n\nResponse:\nThere are originally 3 cars. 2 more cars arrive. \n3 + 2 = 5.\nAnswer: 5\n\nExample question 3: Leah had 32 chocolates and her sister had 42. If they ate 35, how many pieces do they have left in total?\n\nRespo

In [15]:
unprocessed_math_dapo = load_dataset("parquet", data_files="/home/hieunt/VIP/data/vip-dapo-math-17k.parquet")["train"]


Generating train split: 17398 examples [00:00, 160276.62 examples/s]


Dataset({
    features: ['data_source', 'prompt', 'ability', 'reward_model', 'extra_info'],
    num_rows: 83
})

Dataset({
    features: ['question', 'answer', 'difficulty', 'data_source', 'prompt', 'ability', 'reward_model', 'extra_info'],
    num_rows: 83
})

In [17]:
unprocessed_math_dapo['question']

Column(['In triangle $ABC$, $\\sin \\angle A = \\frac{4}{5}$ and $\\angle A < 90^\\circ$. Let $D$ be a point outside triangle $ABC$ such that $\\angle BAD = \\angle DAC$ and $\\angle BDC = 90^\\circ$. Suppose that $AD = 1$ and that $\\frac{BD}{CD} = \\frac{3}{2}$. If $AB + AC$ can be expressed in the form $\\frac{a\\sqrt{b}}{c}$ where $a, b, c$ are pairwise relatively prime integers, find $a + b + c$.', 'Let $ABCD$ be a unit square in the plane. Points $X$ and $Y$ are chosen independently and uniformly at random on the perimeter of $ABCD$. If the expected value of the area of triangle $\\triangle AXY$ can be expressed as $\\frac{m}{n}$ for relatively prime positive integers $m$ and $n$, compute $m+n$.', 'Let $a, b, c$ be distinct numbers such that the equations $x^2 + ax + 1 = 0$ and $x^2 + bx + c = 0$ have a common real root, and the equations $x^2 + x + a = 0$ and $x^2 + cx + b = 0$ also have a common real root. Compute the sum $a + b + c$.', 'There are $7$ boxes arranged in a row an